In [2]:
!git clone https://github.com/YagoBaltazar/Indice_Pobreza_Consolidado
%cd Indice_Pobreza_Consolidado

Cloning into 'Indice_Pobreza_Consolidado'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 32 (delta 5), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 24.80 KiB | 12.40 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/Indice_Pobreza_Consolidado/Indice_Pobreza_Consolidado


# Análise Exploratória — Índices de Pobreza no Brasil

Fonte: [Kaggle](https://www.kaggle.com/datasets/fidelissauro/indices-pobreza-brasil).

O arquivo `indices_pobreza_consolidado.csv` é uma **série temporal mensal nacional** (uma linha por mês, ago/2012 a set/2022), sem dimensão de estado ou região. Por isso a análise é feita **ao longo do tempo** e **entre grupos populacionais** (famílias, indígenas, quilombolas e ciganos), e não entre localidades.

**Perguntas de pesquisa**
- **RQ1:** como evoluíram a pobreza e a extrema pobreza ao longo do período, e os grupos populacionais seguiram o mesmo padrão?
- **RQ2:** como os indicadores se relacionam entre si?

Rode as células em ordem.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px

pd.options.display.float_format = "{:,.2f}".format

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ASSETS = ROOT / "docs" / "assets"
ASSETS.mkdir(parents=True, exist_ok=True)

# procura o CSV: primeiro nos lugares comuns, depois em qualquer subpasta do projeto
NOME = "indices_pobreza_consolidado.csv"
candidatos = [Path.cwd() / NOME, ROOT / NOME, ROOT / "data" / NOME,
              ROOT / "data" / "raw" / NOME, ROOT / "data" / "processed" / NOME]
CSV = next((p for p in candidatos if p.exists()), None) or next(ROOT.rglob(NOME), None)

if CSV is None:
    raise FileNotFoundError(
        f"Não achei {NOME}. Pasta atual: {Path.cwd()}. "
        "Coloque o CSV nessa pasta (ou em data/) e rode de novo. "
        "No Colab/Kaggle, envie o arquivo pelo painel de arquivos à esquerda."
    )
print("Usando:", CSV)

Usando: /content/Indice_Pobreza_Consolidado/Indice_Pobreza_Consolidado/indices_pobreza_consolidado.csv


## 1. Dados brutos

In [4]:
raw = pd.read_csv(CSV)
display(raw.head())
raw.info()

,referencia,periodo,pobreza,extrema_pobreza,total,populacao_estimada,porcentagem_pobreza,porcentagem_extrema_pobreza,porcentagem_vulnerabilidade,familias_pobreza,...,familias_vulnerabilidade,indigenas_pobreza,indigenas_extrema_pobreza,indigenas_vulnerabilidade,quilombolas_pobreza,quilombolas_extrema_pobreza,quilombolas_vulnerabilidade,ciganos_pobreza,ciganos_extrema_pobreza,ciganos_vulnerabilidade
0,08/2012,201208,19370363,44381455,63751818,193976530,0.10,0.23,0.33,5265743,...,18162964,14144,86593,100737,7927,57407,65334,119,757,876
1,09/2012,201209,19371279,44554715,63925994,193976530,0.10,0.23,0.33,5278170,...,18284925,14157,87216,101373,8022,58018,66040,123,825,948
2,10/2012,201210,19376665,44652093,64028758,193976530,0.10,0.23,0.33,5288902,...,18365268,14171,87397,101568,8081,58468,66549,131,888,1019
3,11/2012,201211,19381040,44723215,64104255,193976530,0.10,0.23,0.33,5298295,...,18429654,14182,87697,101879,8198,59193,67391,142,925,1067
4,12/2012,201212,19392882,44684759,64077641,193976530,0.10,0.23,0.33,5311337,...,18472436,14209,87820,102029,8314,60250,68564,143,978,1121


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122 entries, 0 to 121
Data columns (total 21 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   referencia                   122 non-null    object 
 1   periodo                      122 non-null    int64  
 2   pobreza                      122 non-null    int64  
 3   extrema_pobreza              122 non-null    int64  
 4   total                        122 non-null    int64  
 5   populacao_estimada           122 non-null    int64  
 6   porcentagem_pobreza          122 non-null    float64
 7   porcentagem_extrema_pobreza  122 non-null    float64
 8   porcentagem_vulnerabilidade  122 non-null    float64
 9   familias_pobreza             122 non-null    int64  
 10  familias_extrema_pobreza     122 non-null    int64  
 11  familias_vulnerabilidade     122 non-null    int64  
 12  indigenas_pobreza            122 non-null    int64  
 13  indigenas_extrema_po

## 2. Limpeza e validação
Passos aplicados: remoção de duplicatas, conversão de `periodo` (AAAAMM) para data, ordenação cronológica e checagens de consistência. O relatório mostra o que foi encontrado.

In [5]:
df = raw.drop_duplicates().copy()
df["data"] = pd.to_datetime(df["periodo"].astype(str), format="%Y%m")
df = df.sort_values("data").reset_index(drop=True)
df["ano"] = df["data"].dt.year

# colunas de grupo: <grupo>_pobreza / _extrema_pobreza / _vulnerabilidade
GRUPOS = ["familias", "indigenas", "quilombolas", "ciganos"]
for g in GRUPOS:
    df[f"{g}_vulnerabilidade_calc"] = df[f"{g}_pobreza"] + df[f"{g}_extrema_pobreza"]

# percentuais recalculados (o CSV traz só 2 casas decimais, ex.: 0.04, o que perde precisão)
df["pct_pobreza"] = df["pobreza"] / df["populacao_estimada"] * 100
df["pct_extrema_pobreza"] = df["extrema_pobreza"] / df["populacao_estimada"] * 100
df["pct_vulnerabilidade"] = df["total"] / df["populacao_estimada"] * 100

meses_esperados = pd.date_range(df["data"].min(), df["data"].max(), freq="MS")
report = {
    "linhas_brutas": len(raw),
    "duplicatas_removidas": len(raw) - len(df),
    "nulos_totais": int(raw.isna().sum().sum()),
    "periodo": f"{df['data'].min():%m/%Y} a {df['data'].max():%m/%Y}",
    "meses_faltando": len(meses_esperados.difference(df["data"])),
    "referencia_bate_com_periodo": bool((raw["referencia"].values == pd.to_datetime(raw["periodo"].astype(str), format="%Y%m").dt.strftime("%m/%Y").values).all()),
    "total == pobreza + extrema_pobreza": bool((df["total"] == df["pobreza"] + df["extrema_pobreza"]).all()),
    "grupos: vulnerabilidade == pobreza + extrema": all(
        bool((df[f"{g}_vulnerabilidade"] == df[f"{g}_vulnerabilidade_calc"]).all()) for g in GRUPOS
    ),
}
df = df.drop(columns=[f"{g}_vulnerabilidade_calc" for g in GRUPOS])
pd.Series(report, name="relatório").to_frame()

,relatório
linhas_brutas,122
duplicatas_removidas,0
nulos_totais,0
periodo,08/2012 a 09/2022
meses_faltando,0
referencia_bate_com_periodo,True
total == pobreza + extrema_pobreza,True
grupos: vulnerabilidade == pobreza + extrema,True


## 3. Visão geral

In [6]:
num_cols = [c for c in df.select_dtypes(include=np.number).columns if c not in ("periodo", "ano")]
print(f"{df.shape[0]} meses, {len(num_cols)} variáveis numéricas")
display(df[num_cols].describe().T)
display(df.isna().mean().sort_values(ascending=False).rename("proporção_nulos").head(5).to_frame())

122 meses, 22 variáveis numéricas


,count,mean,std,min,25%,50%,75%,max
pobreza,122.00,"12,948,012.13","4,046,276.47","8,454,642.00","8,978,067.50","12,248,774.50","15,451,770.25","19,393,635.00"
extrema_pobreza,122.00,"41,231,462.55","3,015,699.31","36,803,455.00","38,698,620.25","39,777,429.50","44,200,667.50","49,055,941.00"
total,122.00,"54,179,474.68","6,015,011.20","47,229,326.00","48,892,009.00","52,292,939.50","57,708,982.00","64,396,550.00"
populacao_estimada,122.00,"207,211,210.76","4,830,404.93","193,976,530.00","204,482,867.00","207,660,929.00","211,755,692.00","213,317,639.00"
porcentagem_pobreza,122.00,0.06,0.02,0.04,0.04,0.06,0.08,0.10
porcentagem_extrema_pobreza,122.00,0.20,0.02,0.18,0.19,0.19,0.22,0.23
porcentagem_vulnerabilidade,122.00,0.26,0.03,0.22,0.23,0.25,0.28,0.33
familias_pobreza,122.00,"3,865,710.61","958,263.04","2,746,503.00","2,984,406.50","3,668,067.50","4,435,881.25","5,398,817.00"
familias_extrema_pobreza,122.00,"13,630,662.05","1,621,377.87","11,720,302.00","12,718,069.00","13,209,058.50","13,960,209.00","20,439,159.00"
familias_vulnerabilidade,122.00,"17,496,372.66","1,677,817.52","15,569,480.00","16,261,198.25","16,734,127.00","18,586,139.25","23,934,236.00"


,proporção_nulos
referencia,0.00
periodo,0.00
pobreza,0.00
extrema_pobreza,0.00
total,0.00


## 4. Variáveis de análise
- **TEMPO:** `data` (mensal).
- **INDICADORES:** percentual da população em pobreza, extrema pobreza e vulnerabilidade (soma das duas), calculados como `contagem / populacao_estimada`.
- **METRICA:** `pct_extrema_pobreza`, a de maior magnitude.

In [7]:
TEMPO = "data"
INDICADORES = ["pct_pobreza", "pct_extrema_pobreza", "pct_vulnerabilidade"]
METRICA = "pct_extrema_pobreza"
ROTULOS = {
    "pct_pobreza": "Pobreza",
    "pct_extrema_pobreza": "Extrema pobreza",
    "pct_vulnerabilidade": "Vulnerabilidade (total)",
}

## 5. Distribuição e outliers (IQR)
Como os dados são uma série com tendência, valores fora do intervalo do IQR indicam **meses atípicos em relação ao restante do período**, não erros de medição.

In [8]:
fig1 = px.histogram(df, x=METRICA, nbins=20, marginal="box",
                    labels={METRICA: ROTULOS[METRICA] + " (% da população)"},
                    title=f"Distribuição de {ROTULOS[METRICA].lower()} (% da população)")
fig1.show(); fig1.write_html(ASSETS / "fig1_distribuicao.html", include_plotlyjs="cdn")

q1, q3 = df[METRICA].quantile([.25, .75]); iqr = q3 - q1
outliers = df[(df[METRICA] < q1 - 1.5 * iqr) | (df[METRICA] > q3 + 1.5 * iqr)]
print(f"{len(outliers)} outliers em {METRICA} (limites: {q1 - 1.5*iqr:.2f} a {q3 + 1.5*iqr:.2f})")
outliers[["data", METRICA]]

0 outliers em pct_extrema_pobreza (limites: 13.51 a 26.95)


,data,pct_extrema_pobreza


## 6. RQ1 — evolução ao longo do tempo

In [9]:
long = df.melt(id_vars=TEMPO, value_vars=INDICADORES, var_name="indicador", value_name="pct")
long["indicador"] = long["indicador"].map(ROTULOS)
fig2 = px.line(long, x=TEMPO, y="pct", color="indicador",
               labels={"pct": "% da população", "data": "Mês", "indicador": ""},
               title="Pobreza, extrema pobreza e vulnerabilidade (% da população)")
fig2.show(); fig2.write_html(ASSETS / "fig2_rq1.html", include_plotlyjs="cdn")

### 6.1 Grupos populacionais
Os grupos têm tamanhos muito diferentes, então as séries são colocadas em **índice base 100** (primeiro mês = 100) para comparar a trajetória, não o nível.

In [10]:
vuln = df.set_index(TEMPO)[[f"{g}_vulnerabilidade" for g in GRUPOS]]
vuln.columns = GRUPOS
base100 = (vuln / vuln.iloc[0] * 100).reset_index().melt(id_vars=TEMPO, var_name="grupo", value_name="indice")
fig2b = px.line(base100, x=TEMPO, y="indice", color="grupo",
                labels={"indice": "Índice (ago/2012 = 100)", "data": "Mês", "grupo": ""},
                title="Vulnerabilidade por grupo (índice base 100; famílias contam famílias, os demais contam pessoas)")
fig2b.show(); fig2b.write_html(ASSETS / "fig2b_grupos.html", include_plotlyjs="cdn")

## 7. RQ2 — relação entre indicadores
Correlação em **nível** e em **variação mensal**. Em séries temporais com tendência, a correlação em nível tende a ser inflada por tendência comum; a variação mensal mostra se os indicadores se movem juntos de fato.

In [11]:
cols = ["pobreza", "extrema_pobreza", "total", "populacao_estimada",
        "familias_vulnerabilidade", "indigenas_vulnerabilidade",
        "quilombolas_vulnerabilidade", "ciganos_vulnerabilidade"]
corr_nivel = df[cols].corr()
corr_var = df[cols].pct_change().dropna().corr()

fig3 = px.imshow(corr_nivel, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                 title="Correlação entre indicadores (em nível)")
fig3.show(); fig3.write_html(ASSETS / "fig3_correlacao.html", include_plotlyjs="cdn")

fig3b = px.imshow(corr_var, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                  title="Correlação entre variações mensais")
fig3b.show(); fig3b.write_html(ASSETS / "fig3b_correlacao_variacao.html", include_plotlyjs="cdn")

## 8. Números-chave para os achados

In [12]:
def ponto(col, funcao):
    i = funcao(df[col]); return df.loc[i, "data"].strftime("%m/%Y"), df.loc[i, col]

for col in INDICADORES:
    ini, fim = df[col].iloc[0], df[col].iloc[-1]
    d_max, v_max = ponto(col, pd.Series.idxmax); d_min, v_min = ponto(col, pd.Series.idxmin)
    print(f"{ROTULOS[col]}: {ini:.2f}% -> {fim:.2f}% ({fim - ini:+.2f} p.p.) | mínimo {v_min:.2f}% em {d_min} | máximo {v_max:.2f}% em {d_max}")

print()
print("Variação na contagem em vulnerabilidade (primeiro -> último mês):")
for g in GRUPOS:
    c = f"{g}_vulnerabilidade"; print(f"  {g}: {df[c].iloc[0]:,.0f} -> {df[c].iloc[-1]:,.0f} ({df[c].iloc[-1]/df[c].iloc[0]-1:+.1%})")

print()
par = corr_var.where(np.triu(np.ones(corr_var.shape, dtype=bool), 1)).stack()
print("Par mais correlacionado em variação mensal:", par.idxmax(), f"{par.max():.2f}")
print("Par menos correlacionado em variação mensal:", par.idxmin(), f"{par.min():.2f}")

Pobreza: 9.99% -> 4.46% (-5.53 p.p.) | mínimo 3.96% em 01/2022 | máximo 10.00% em 12/2012
Extrema pobreza: 22.88% -> 23.00% (+0.12 p.p.) | mínimo 17.86% em 12/2016 | máximo 23.06% em 11/2012
Vulnerabilidade (total): 32.87% -> 27.45% (-5.41 p.p.) | mínimo 22.47% em 02/2020 | máximo 33.05% em 11/2012

Variação na contagem em vulnerabilidade (primeiro -> último mês):
  familias: 18,162,964 -> 23,934,236 (+31.8%)
  indigenas: 100,737 -> 185,613 (+84.3%)
  quilombolas: 65,334 -> 207,596 (+217.7%)
  ciganos: 876 -> 9,361 (+968.6%)

Par mais correlacionado em variação mensal: ('total', 'familias_vulnerabilidade') 0.98
Par menos correlacionado em variação mensal: ('populacao_estimada', 'indigenas_vulnerabilidade') -0.13


## 9. Achados
Escreva aqui 3 a 5 frases com os números da seção 8 que respondem RQ1 e RQ2. Elas viram a seção 3 do artigo.

In [13]:
import os
from pathlib import Path

# Define e cria o diretório docs/assets
ASSETS = Path.cwd().parent / "docs" / "assets" if Path.cwd().name == "notebooks" else Path.cwd() / "docs" / "assets"
ASSETS.mkdir(parents=True, exist_ok=True)

# Salva todas as 5 figuras interativas do Plotly
fig1.write_html(ASSETS / "fig1_distribuicao.html", include_plotlyjs="cdn")
fig2.write_html(ASSETS / "fig2_rq1.html", include_plotlyjs="cdn")
fig2b.write_html(ASSETS / "fig2b_grupos.html", include_plotlyjs="cdn")
fig3.write_html(ASSETS / "fig3_correlacao.html", include_plotlyjs="cdn")
fig3b.write_html(ASSETS / "fig3b_correlacao_variacao.html", include_plotlyjs="cdn")

print("Figuras HTML exportadas com sucesso para docs/assets/")

Figuras HTML exportadas com sucesso para docs/assets/
